In [9]:
import os
import yfinance as yf
import time
import smtplib
from email.message import EmailMessage
import pandas as pd 

C = {'G': '\033[92m', 'R': '\033[91m', 'Y': '\033[93m', 'B': '\033[94m', 'C': '\033[96m', 'W': '\033[0m'}
ATR_MULTIPLIER = 2.0 
TRAILING_STOP_PCT = -0.015  

def log_system_event(message):
    timestamp = time.ctime()
    with open('system_log.txt', 'a') as f:
        f.write(f'[{timestamp}], {message}\n')
    
def send_trade_notification(signal, price, pnl):
    msg = EmailMessage()
    msg.set_content(f'Update From The Bot:\nAction: {signal}\nPrice: {price:.2f}\nPnL: {pnl:.2f}')
    msg['Subject'] = f'Bot Alert: {signal}'
    msg['From'] = 'smtp.quant@gmail.com'
    msg['To'] = 'smtp.quant@gmail.com'
    with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
        smtp.login('smtp.quant@gmail.com', 'My Passcode')
        smtp.send_message(msg)

last_buy_price = 0
last_signal = 'None'
highest_price = 0
total_pnl = 0.0
wins, losses = 0, 0

if os.path.exists('trade_log.csv'):
    try:
        df_history = pd.read_csv('trade_log.csv', names=['Time', 'Price', 'Signal', 'PnL'])
        df_history['PnL'] = pd.to_numeric(df_history['PnL'], errors='coerce').fillna(0)
        total_pnl = df_history['PnL'].sum()
        wins = len(df_history[df_history['PnL'] > 0])
        losses = len(df_history[df_history['PnL'] < 0])
        last_signal = str(df_history.iloc[-1]['Signal']).strip()
        if last_signal == 'BUY':
            last_buy_price = float(df_history.iloc[-1]['Price'])
            highest_price = last_buy_price
    except: pass

print(f"{C['B']} Initializing volatility-aware engine...{C['W']}")
master_df = yf.download('RELIANCE.NS', period='5d', interval='1m', progress=False)
print(f"{C['G']}Month 2 Active: ATR Dynamic Risk Management Online{C['W']}\n")
while True:
    try:
        new_data = yf.download('RELIANCE.NS', period='1d', interval='1m', progress=False)
        if not new_data.empty:
            new_candle = new_data.tail(1) 
            master_df = pd.concat([master_df, new_candle])
            master_df = master_df[~master_df.index.duplicated(keep='last')].tail(250)
        else:
            time.sleep(10); continue
        master_df['SMA_45'] = master_df['Close']['RELIANCE.NS'].rolling(window=45).mean()
        master_df['SMA_190'] = master_df['Close']['RELIANCE.NS'].rolling(window=190).mean()
        delta = master_df['Close']['RELIANCE.NS'].diff()
        gain = delta.clip(lower=0)
        loss = delta.clip(upper=0).abs()
        master_df['RSI'] = 100 - (100 / (1 + (gain.rolling(14).mean() / loss.rolling(14).mean())))
        master_df['EMA_12'] = master_df['Close']['RELIANCE.NS'].ewm(span=12, adjust=False).mean()
        master_df['EMA_26'] = master_df['Close']['RELIANCE.NS'].ewm(span=26, adjust=False).mean()
        master_df['MACD'] = master_df['EMA_12'] - master_df['EMA_26']
        master_df['MACD_Signal'] = master_df['MACD'].ewm(span=9, adjust=False).mean()
        high = master_df['High']['RELIANCE.NS']
        low = master_df['Low']['RELIANCE.NS']
        pc = master_df['Close']['RELIANCE.NS'].shift(1)
        tr = pd.concat([high-low, (high-pc).abs(), (low-pc).abs()], axis=1).max(axis=1)
        master_df['ATR'] = tr.rolling(window=14).mean()
        latest = master_df.iloc[-1]
        current_price = latest['Close']['RELIANCE.NS'].item()
        current_rsi = latest['RSI'].item()
        sma_45, sma_190 = latest['SMA_45'].item(), latest['SMA_190'].item()
        current_macd, current_macd_signal = latest['MACD'].item(), latest['MACD_Signal'].item()
        current_atr = latest['ATR'].item()
        if last_signal == 'BUY' and last_buy_price > 0:
            dynamic_stop_loss = last_buy_price - (current_atr * ATR_MULTIPLIER)            
            if current_price > highest_price: highest_price = current_price
            drawdown = (current_price - highest_price) / highest_price          
            trigger = None
            if current_price <= dynamic_stop_loss: trigger = 'ATR_STOP_LOSS'
            elif drawdown <= TRAILING_STOP_PCT: trigger = 'TRAILING_STOP'           
            if trigger:
                pnl = current_price - last_buy_price
                total_pnl += pnl
                if pnl > 0: wins += 1 
                else: losses += 1   
                print(f"\n{C['R']} {trigger} HIT! Price: {current_price:.2f} | Trade PnL: {pnl:.2f}{C['W']}")
                with open('trade_log.csv', 'a') as f:
                    f.write(f'{time.ctime()},{current_price},{trigger},{pnl}\n')
                
                try: send_trade_notification(trigger, current_price, pnl)
                except: pass
                
                last_signal, last_buy_price, highest_price = 'COOLDOWN', 0, 0
                time.sleep(60); continue
        if (sma_45 > sma_190) and (current_rsi < 60) and (current_macd > current_macd_signal):
            current_signal = 'BUY'
        else:
            current_signal = 'SELL'
        if last_signal == 'COOLDOWN' and current_signal == 'SELL': last_signal = 'SELL'
        elif current_signal != last_signal:
            pnl = 0
            if current_signal == 'SELL' and last_buy_price > 0:
                pnl = current_price - last_buy_price
                total_pnl += pnl
                if pnl > 0: wins += 1 
                else: losses += 1
                print(f"\n{C['Y']} Exit at {current_price:.2f} | PnL: {pnl:.2f}{C['W']}")
            elif current_signal == 'BUY':
                last_buy_price = highest_price = current_price
                print(f"\n{C['G']}Entry at {current_price:.2f} (ATR: {current_atr:.2f}){C['W']}")
            
            with open('trade_log.csv','a') as f:
                f.write(f'{time.ctime()},{current_price},{current_signal},{pnl}\n')
            try: send_trade_notification(current_signal, current_price, pnl)
            except: pass
            last_signal = current_signal
        pnl_val = current_price - last_buy_price if last_signal == 'BUY' else 0
        p_col = C['G'] if pnl_val >= 0 else C['R']
        print(f"\r{C['B']}[{time.strftime('%H:%M:%S')}] {C['W']}"
              f"Prc: {current_price:.2f} | RSI: {current_rsi:.1f} | "
              f"ATR Stop: {C['Y']}{ (last_buy_price - (current_atr * ATR_MULTIPLIER)):.2f}{C['W']} | "
              f"Trade: {p_col}{pnl_val:+.2f}{C['W']} | Total: {total_pnl:+.2f} | W/L: {wins}/{losses}    ", end="")

    except Exception as e:
        print(f"\n{C['R']} Error: {e}{C['W']}"); log_system_event(str(e)); time.sleep(15)

    time.sleep(60)

 Initializing volatility-aware engine...
Month 2 Active: ATR Dynamic Risk Management Online

[21:01:02] Prc: 1339.00 | RSI: 69.8 | ATR Stop: -1.71 | Trade: +0.00 | Total: -63.30 | W/L: 0/3    

KeyboardInterrupt: 